# Benchmark - run a grid on Colab
Plays the games of a grid (`grids/*.yaml`) that have not been played yet, one model at a time, and saves one record per game on Google Drive.
The games already played are skipped: the new ones on Drive and the old experiments imported in `results/` (see `import_legacy.py`).
If the run stops (or the runtime is deleted), run the notebook again: it resumes from the games not yet saved.

Recommended order: a pilot of a few games per new model (`limit = 8`) to see the speed on the L4, then the whole `tier1.yaml`, then `tier1_families.yaml`.

# Setup
Clones the repository (only the folders needed), installs TextWorld and vLLM and moves into `Code/Benchmark`. Locally it does nothing.
The first time on a new Colab machine the installation takes a few minutes: restart the kernel after it (and then run all the cells), because it can update torch.

In [1]:
import os
import sys
import shutil
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/UPROOT01/LLMs-Play-Textual-Games.git"
    repo_dir = "/content/LLMs-Play-Textual-Games"
    if not os.path.isfile(f"{repo_dir}/Code/Benchmark/run.py"):
        shutil.rmtree(repo_dir, ignore_errors=True) # leftover of a failed clone, or a clone without the benchmark
        # sparse clone: only the folders needed (the whole Games folder is >1 GB)
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", repo_dir, "sparse-checkout", "add", "Code/Benchmark",
                    "Code/Games/cooking", "Code/Games/coin_collector", "Code/Games/treasure_hunter"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "textworld", "vllm==0.31.0", "pyyaml"], check=True)
    os.chdir(f"{repo_dir}/Code/Benchmark")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

Working directory: /content/LLMs-Play-Textual-Games/Code/Benchmark


In [2]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import torch

assert torch.cuda.is_available(), "No GPU available: connect to a Colab runtime with a GPU (e.g. L4)"
! nvidia-smi

Wed Oct 21 12:49:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   48C    P8             14W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# What to run
- `grid`: the grid file. `models`: only these models of the grid (`None`: all of them, one after the other).
- `limit`: at most this many games per model (`None`: all). Use it for a pilot.
- `n_parallel`: games played at the same time. vLLM answers them together: more games, more tokens per minute, until the GPU memory is full.
  `"auto"` decides by itself: it starts a new game only if the contexts of all the games, at the size they are expected to reach, fit in the GPU memory (see `engine.ContextBudget`).
  The status line shows how full the GPU memory is expected to be. A number (e.g. `8`) plays always that many.

To change the models or their vLLM settings (e.g. a smaller `max_model_len` if a model does not fit), see `profiles.py`.

In [5]:
from records import LEGACY_RESULTS
from run import load_grid, run_grid, summary

grid = load_grid("grids/tier1.yaml")
# models = ["Qwen/Qwen3-1.7B"] # None: all the models of the grid
models = None
limit = None # e.g. 8 for a pilot
n_parallel = "auto"

if IN_COLAB:
    assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not mounted: run the cell above"
    results = "/content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results"
else:
    results = "results_new"
read = [LEGACY_RESULTS] # the old experiments: their games are not played again
print("Results folder:", results)
summary(grid, results, read, models)

Results folder: /content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results
model                                          mode  played  left  ~Mtokens left
Qwen/Qwen3-1.7B                                   0      50     0            0.0
Qwen/Qwen3-1.7B                                  e1      50     0            0.0
Qwen/Qwen3-1.7B                                  r4      50     0            0.0
Qwen/Qwen3-1.7B                                 er4      50     0            0.0
Qwen/Qwen3-1.7B                                 lr4      50     0            0.0
Qwen/Qwen3-8B                                     0       0    50            0.1
Qwen/Qwen3-8B                                    e1       0    50            2.5
Qwen/Qwen3-8B                                    r4       0    50            0.3
Qwen/Qwen3-8B                                   er4       0    50            0.3
Qwen/Qwen3-8B                                   lr4       0    50            0.4
Qwen/Qwen3-4B               

# Run
Every game is saved as soon as it ends. Stopping the cell is safe: the games being played are lost, the others are kept.
If loading the second model of a run fails because the GPU memory is full, restart the runtime and run everything again: the first model's games are skipped.

In [6]:
run_grid(grid, results, read, models=models, limit=limit, n_parallel=n_parallel)

Qwen/Qwen3-1.7B: 0 games to play
Qwen/Qwen3-8B: 250 games to play


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


[rank0]:W1021 13:01:27.155000 4306 torch/_inductor/utils.py:1953] Not enough SMs to use max_autotune_gemm mode
Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00,  8.74it/s]


ValueError: To serve at least one request with the model's max seq len (40960), (2.81 GiB KV cache is needed, which is larger than the available KV cache memory (2.17 GiB). Based on the available memory, the estimated maximum model length is 31568. Try increasing `gpu_memory_utilization` (which also controls CPU memory on the CPU backend) or decreasing `max_model_len` when initializing the engine. See https://docs.vllm.ai/en/latest/configuration/conserving_memory/ for more details.

# Quick look
Average normalized score, games and thousands of generated tokens per game, per model and mode (the graphs are in `analysis.ipynb`).

In [ ]:
from records import load_table

table = load_table(results, LEGACY_RESULTS)
games = table[table.game.isin(grid["games"]) & table.prompt.isin(grid["prompts"]) & table.seed.isin(grid["seeds"])
              & table.model.isin(grid["models"]) & table["mode"].isin(grid["modes"])]
games = games[[level in grid["games"][game] for game, level in zip(games.game, games.level)]]
games.groupby(["model", "mode"]).agg(score=("norm_score", "mean"), games=("seed", "size"),
                                     ktokens=("tokens_generated", lambda t: t.mean() / 1000)).round(2)

score  games  ktokens
model           mode                       
Qwen/Qwen3-1.7B 0      0.02     50     1.62
                e1     0.21     50    42.93
                er4    0.04     50     2.43
                lr4    0.06     50     2.97
                r4     0.05     50     5.07
Qwen/Qwen3-4B   e1     0.47     50    61.65
                er4    0.35     50    10.92
                lr4    0.42     50    11.68
                r4     0.24     50      NaN